In [1]:
# ============================================================
# SETUP PATHS & IMPORTS
# ============================================================

import os
import sys
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Local path routing
# ------------------------------------------------------------
current_dir = os.getcwd()
workspace_root = current_dir

if os.path.basename(current_dir) == "notebooks":
    workspace_root = os.path.dirname(current_dir)

src_path = os.path.join(workspace_root, "src")

sys.path.insert(0, workspace_root)
sys.path.insert(0, src_path)

# ------------------------------------------------------------
# Kaggle path routing
# ------------------------------------------------------------
kaggle_input = Path("/kaggle/input")

if kaggle_input.exists():
    for py_file in kaggle_input.rglob("*.py"):
        sys.path.insert(0, str(py_file.parent))

# ------------------------------------------------------------
# sklearn imports
# ------------------------------------------------------------
from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedShuffleSplit,
    train_test_split,
    GridSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.ensemble import VotingClassifier, RandomForestClassifier

# ------------------------------------------------------------
# XGBoost import
# ------------------------------------------------------------
try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("⚠️ XGBoost not found. Please install it via `pip install xgboost`.")

# ------------------------------------------------------------
# Optional Bayesian search imports
# ------------------------------------------------------------
try:
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer
    SKOPT_AVAILABLE = True
except Exception:
    BayesSearchCV = None
    Categorical = None
    Integer = None
    SKOPT_AVAILABLE = False

# ------------------------------------------------------------
# Project imports
# ------------------------------------------------------------
try:
    from src.ev_data_utils import load_ev_data
    from src.ev_baseline_utils import (
        EVFeatureExtractor,
        LinearBaselineClassifier,
        competition_score,
        make_competition_scorer,
    )
    print("✅ Imports loaded from src/")
except ImportError:
    from ev_data_utils import load_ev_data
    from ev_baseline_utils import (
        EVFeatureExtractor,
        LinearBaselineClassifier,
        competition_score,
        make_competition_scorer,
    )
    print("✅ Imports loaded flattened")

print(f"SKOPT_AVAILABLE: {SKOPT_AVAILABLE}")
print(f"XGB_AVAILABLE: {XGB_AVAILABLE}")

✅ Imports loaded flattened
SKOPT_AVAILABLE: True
XGB_AVAILABLE: True


In [2]:
# ============================================================
# CONFIGURATION
# ============================================================

TARGET_COL = "Will_Buy_EV"

RANDOM_STATE = 42
N_SPLITS = 1                # Set to 1 for a single stratified fold, or >= 2 for K-Fold
HOLDOUT_SIZE = 0.4

MAKE_SUBMISSION = True

# ------------------------------------------------------------
# Search configuration
# ------------------------------------------------------------
SEARCH_MODE = "bayesian"        # "grid" or "bayesian"
N_ITER = 10               # used only for Bayesian search
VERBOSE = 3
ERROR_SCORE = "raise"       # use "raise" for debugging

USE_BAYESIAN = SEARCH_MODE == "bayesian" and SKOPT_AVAILABLE

if SEARCH_MODE == "bayesian" and not SKOPT_AVAILABLE:
    print("⚠️ search_mode='bayesian' but scikit-optimize is unavailable.")
    print("⚠️ Falling back to grid search.")

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M")

print(f"Search mode: {SEARCH_MODE}")
print(f"Using Bayesian search: {USE_BAYESIAN}")

Search mode: bayesian
Using Bayesian search: True


In [3]:
# ============================================================
# DATA LOADING
# ============================================================

train_df, test_df, data_source = load_ev_data(
    local_dir="data",
    train_name="train.csv",
    test_name="test.csv",
    sample_name="sample_ev.csv",
    target_col=TARGET_COL,
)

print(f"✅ Data source: {data_source}")
print(f"Train shape: {train_df.shape}")

if test_df is not None:
    print(f"Test shape: {test_df.shape}")

# ------------------------------------------------------------
# Target check
# ------------------------------------------------------------
if TARGET_COL not in train_df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found in train data.")

train_df = train_df.dropna(subset=[TARGET_COL]).copy()
train_df[TARGET_COL] = train_df[TARGET_COL].astype(int)

print("\nTarget distribution:")
print(train_df[TARGET_COL].value_counts())

assert set(train_df[TARGET_COL].unique()).issubset({0, 1}), (
    "Target must be binary 0/1. "
    "Coercion should happen inside ev_data_utils.load_ev_data."
)

# ------------------------------------------------------------
# Features / target
# ------------------------------------------------------------
X = train_df.drop(columns=[TARGET_COL], errors="ignore").copy()
y = train_df[TARGET_COL].astype(int).copy()

# ------------------------------------------------------------
# Optional holdout split
# ------------------------------------------------------------
if len(X) >= 50 and y.nunique() > 1:
    X_train, X_holdout, y_train, y_holdout = train_test_split(
        X,
        y,
        test_size=HOLDOUT_SIZE,
        stratify=y,
        random_state=RANDOM_STATE,
    )
    DO_HOLDOUT = True
else:
    X_train = X.copy()
    y_train = y.copy()
    X_holdout = None
    y_holdout = None
    DO_HOLDOUT = False

print(f"\nTrain rows used for fitting/search: {len(X_train)}")

if DO_HOLDOUT:
    print(f"Holdout rows: {len(X_holdout)}")
else:
    print("No holdout split.")

✅ Using Kaggle data folder: /kaggle/input/competitions/playground-series-s6e9
✅ Data source: Train file: /kaggle/input/competitions/playground-series-s6e9/train.csv
Train shape: (668665, 15)
Test shape: (286571, 14)

Target distribution:
Will_Buy_EV
0    551886
1    116779
Name: count, dtype: int64

Train rows used for fitting/search: 401199
Holdout rows: 267466


In [4]:
# ============================================================
# ENSEMBLE PIPELINE CREATION
# ============================================================

if not XGB_AVAILABLE:
    raise RuntimeError("XGBoost is required for this ensemble notebook. Please install it.")

# Shared Multivariate Feature Extractor
# Note: scale_numeric=True is required because Linear Regression needs scaled features.
# Tree-based models (XGBoost, RF) are invariant to scaling, so they won't be negatively affected.
shared_extractor = EVFeatureExtractor(
    feature_set="multivariate",
    target_col=TARGET_COL,
    drop_id=True,
    impute_strategy="median",
    scale_numeric=True,
    onehot_categorical=True,
    add_derived_features=True,
)

# Base Estimators
lr_model = LinearBaselineClassifier(
    model_type="linear_regression",
    fit_intercept=True,
    clip_predictions=True,
    random_state=RANDOM_STATE,
)

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    eval_metric="logloss",
)

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

# Ensemble Model (Soft Voting)
ensemble_model = VotingClassifier(
    estimators=[
        ("lr", lr_model),
        ("xgb", xgb_model),
        ("rf", rf_model),
    ],
    voting="soft",
    weights=(1, 1, 1),
    n_jobs=-1,
)

# Final Pipeline
ensemble_pipeline = Pipeline(
    [
        ("extractor", shared_extractor),
        ("model", ensemble_model),
    ]
)

print("✅ Created Multivariate Ensemble Pipeline (Linear Regression + XGBoost + Random Forest)")

✅ Created Multivariate Ensemble Pipeline (Linear Regression + XGBoost + Random Forest)


In [5]:
# ============================================================
# COMPETITION SCORER
# ============================================================

scorer = make_competition_scorer(target_col=TARGET_COL)

print("✅ Using ROC AUC competition scorer.")

✅ Using ROC AUC competition scorer.


In [6]:
# ============================================================
# GRID SEARCH SPACE
# ============================================================

ENSEMBLE_GRID_SPACE = {
    # Feature extraction parameters
    "extractor__add_derived_features": [True],
    "extractor__impute_strategy": ["median"],

    # Ensemble weights
    "model__weights": [(1, 1, 1)],

    # Linear Regression parameters
    "model__lr__clip_predictions": [True, False],

    # XGBoost parameters
    "model__xgb__n_estimators": [100],
    "model__xgb__max_depth": [4],

    # Random Forest parameters
    "model__rf__n_estimators": [100],
    "model__rf__max_depth": [None, 15],
}

# ============================================================
# BAYESIAN SEARCH SPACE
# ============================================================

if SKOPT_AVAILABLE:
    ENSEMBLE_BAYESIAN_SPACE = {
        # Feature extraction parameters
        "extractor__add_derived_features": Categorical([True, False]),
        "extractor__impute_strategy": Categorical(["median", "mean"]),

        # Linear Regression parameters
        "model__lr__clip_predictions": Categorical([True, False]),

        # XGBoost parameters
        "model__xgb__n_estimators": Integer(10, 200),
        "model__xgb__max_depth": Integer(3, 10),

        # Random Forest parameters
        "model__rf__n_estimators": Integer(10, 200),
        "model__rf__max_depth": Categorical([None, 10, 20, 30]),
    }
else:
    ENSEMBLE_BAYESIAN_SPACE = None

# ============================================================
# SELECT ACTIVE SEARCH SPACE
# ============================================================

if USE_BAYESIAN:
    ENSEMBLE_SEARCH_SPACE = ENSEMBLE_BAYESIAN_SPACE
    SEARCH_SPACE_KIND = "Bayesian"
else:
    ENSEMBLE_SEARCH_SPACE = ENSEMBLE_GRID_SPACE
    SEARCH_SPACE_KIND = "Grid"

print(f"✅ Active search-space type: {SEARCH_SPACE_KIND}")

✅ Active search-space type: Bayesian


In [7]:
# ============================================================
# CV OBJECT
# ============================================================

CAN_CV = (
    y_train.nunique() > 1
    and len(y_train) >= 4
    and int(y_train.value_counts().min()) >= 2
)

if CAN_CV:
    min_class_count = int(y_train.value_counts().min())
    
    if N_SPLITS == 1:
        # StratifiedKFold requires n_splits >= 2.
        # For a single stratified fold, we use StratifiedShuffleSplit.
        cv_object = StratifiedShuffleSplit(
            n_splits=1,
            test_size=HOLDOUT_SIZE,
            random_state=RANDOM_STATE,
        )
        print(f"✅ Using StratifiedShuffleSplit (1 stratified fold).")
    else:
        n_splits = max(2, min(N_SPLITS, min_class_count))
        cv_object = StratifiedKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=RANDOM_STATE,
        )
        print(f"✅ Using StratifiedKFold with {n_splits} folds.")

else:
    cv_object = None
    print("⚠️ Not enough class diversity for stratified CV.")
    print("⚠️ Falling back to direct fit without search.")

✅ Using StratifiedShuffleSplit (1 stratified fold).


In [8]:
# ============================================================
# SEARCH EXECUTION
# ============================================================

if CAN_CV:

    if USE_BAYESIAN:
        ensemble_search = BayesSearchCV(
            estimator=ensemble_pipeline,
            search_spaces=ENSEMBLE_SEARCH_SPACE,
            n_iter=N_ITER,
            scoring=scorer,
            cv=cv_object,
            n_jobs=1,
            random_state=RANDOM_STATE,
            verbose=VERBOSE,
            refit=True,
            return_train_score=True,
            error_score=ERROR_SCORE,
        )
    else:
        ensemble_search = GridSearchCV(
            estimator=ensemble_pipeline,
            param_grid=ENSEMBLE_SEARCH_SPACE,
            scoring=scorer,
            cv=cv_object,
            n_jobs=1,
            verbose=VERBOSE,
            refit=True,
            return_train_score=True,
            error_score=ERROR_SCORE,
        )

    print(f"🚀 Running {SEARCH_SPACE_KIND} search for Ensemble model...")
    ensemble_search.fit(X_train, y_train)

    BEST_PIPELINE = ensemble_search.best_estimator_
    SEARCH_SCORE = float(ensemble_search.best_score_)

    print("\nBest params:")
    for k, v in ensemble_search.best_params_.items():
        print(f"  {k}: {v}")

else:
    print("⚠️ Skipping search and fitting pipeline directly.")
    ensemble_pipeline.fit(X_train, y_train)
    ensemble_search = None
    BEST_PIPELINE = ensemble_pipeline
    SEARCH_SCORE = competition_score(
        y_train,
        BEST_PIPELINE.predict_proba(X_train)[:, 1],
    )

search_results = pd.DataFrame(
    [
        {
            "model": "ensemble_lr_xgb_rf",
            "search_type": SEARCH_SPACE_KIND,
            "search_score_roc_auc": SEARCH_SCORE,
        }
    ]
)

print("\nSearch Results:")
print(search_results)

🚀 Running Bayesian search for Ensemble model...
Fitting 1 folds for each of 1 candidates, totalling 1 fits
[CV 1/1] END extractor__add_derived_features=True, extractor__impute_strategy=mean, model__lr__clip_predictions=False, model__rf__max_depth=10, model__rf__n_estimators=137, model__xgb__max_depth=6, model__xgb__n_estimators=77;, score=(train=0.943, test=0.936) total time=  35.9s
Fitting 1 folds for each of 1 candidates, totalling 1 fits
[CV 1/1] END extractor__add_derived_features=False, extractor__impute_strategy=mean, model__lr__clip_predictions=True, model__rf__max_depth=30, model__rf__n_estimators=174, model__xgb__max_depth=3, model__xgb__n_estimators=36;, score=(train=0.990, test=0.936) total time=  44.3s
Fitting 1 folds for each of 1 candidates, totalling 1 fits
[CV 1/1] END extractor__add_derived_features=True, extractor__impute_strategy=mean, model__lr__clip_predictions=True, model__rf__max_depth=10, model__rf__n_estimators=46, model__xgb__max_depth=6, model__xgb__n_estimat

In [9]:
# ============================================================
# HOLDOUT EVALUATION
# ============================================================

if DO_HOLDOUT:
    ensemble_holdout_score = competition_score(
        y_holdout,
        BEST_PIPELINE.predict_proba(X_holdout)[:, 1],
    )

    print("\nHoldout ROC AUC:")
    print(f"Ensemble: {ensemble_holdout_score:.4f}")

    FINAL_SCORE = ensemble_holdout_score
else:
    FINAL_SCORE = SEARCH_SCORE

print(f"\n🏆 Final ROC AUC Score: {FINAL_SCORE:.4f}")


Holdout ROC AUC:
Ensemble: 0.9399

🏆 Final ROC AUC Score: 0.9399


In [10]:
# ============================================================
# SAVE CV RESULTS WITH PARAMETERS
# ============================================================

if 'ensemble_search' in locals() and ensemble_search is not None:
    ensemble_cv_df = pd.DataFrame(ensemble_search.cv_results_)
    ensemble_cv_df['model_architecture'] = 'ensemble_lr_xgb_rf'
    
    # Move the architecture and score columns to the front for readability
    cols_to_front = ['model_architecture', 'mean_test_score', 'std_test_score', 'params']
    other_cols = [c for c in ensemble_cv_df.columns if c not in cols_to_front]
    ensemble_cv_df = ensemble_cv_df[cols_to_front + other_cols]
    
    cv_path = results_dir / f"ensemble_cv_results_with_params_{timestamp}.csv"
    ensemble_cv_df.to_csv(cv_path, index=False)
    print(f"✅ CV results with parameters saved to: {cv_path}")
else:
    print("⚠️ No search object found. Skipped saving CV results.")

✅ CV results with parameters saved to: results/ensemble_cv_results_with_params_20260914_0335.csv


In [11]:
# ============================================================
# HOLDOUT ERROR ANALYSIS BY PROMINENT CATEGORIES
# ============================================================

if DO_HOLDOUT and X_holdout is not None:
    print("🚀 Generating holdout error analysis...")
    
    # 1. Get predictions from the best pipeline
    holdout_proba = BEST_PIPELINE.predict_proba(X_holdout)[:, 1]
    holdout_preds = (holdout_proba >= 0.5).astype(int)
    
    # 2. Create a master evaluation dataframe
    eval_df = X_holdout.copy()
    eval_df['true_label'] = y_holdout.values
    eval_df['pred_label'] = holdout_preds
    eval_df['pred_proba'] = holdout_proba
    eval_df['is_correct'] = eval_df['true_label'] == eval_df['pred_label']
    
    # 3. Define the prominent categorical/ordinal features to analyze
    prominent_categories = [
        'Gender', 
        'City_Type', 
        'Current_Car_Type', 
        'Home_Charging_Possible', 
        'Subsidy_Available', 
        'Range_Anxiety_Level'
    ]
    
    # Filter to only categories that actually exist in the dataset
    valid_categories = [c for c in prominent_categories if c in eval_df.columns]
    
    breakdown_list = []
    
    # 4. Group by each category and calculate correct/wrong metrics
    for cat in valid_categories:
        grp = eval_df.groupby(cat).agg(
            total_samples=('is_correct', 'count'),
            correct_predictions=('is_correct', 'sum'),
            wrong_predictions=('is_correct', lambda x: (x == False).sum()),
            accuracy=('is_correct', 'mean'),
            actual_yes_count=('true_label', 'sum'),
            predicted_yes_count=('pred_label', 'sum')
        ).reset_index()
        
        grp['feature_name'] = cat
        grp.rename(columns={cat: 'feature_value'}, inplace=True)
        breakdown_list.append(grp)
        
    if breakdown_list:
        breakdown_df = pd.concat(breakdown_list, ignore_index=True)
        
        # Reorder columns for easy reading
        cols = [
            'feature_name', 'feature_value', 'total_samples', 
            'correct_predictions', 'wrong_predictions', 'accuracy', 
            'actual_yes_count', 'predicted_yes_count'
        ]
        breakdown_df = breakdown_df[cols]
        
        # Save the breakdown to CSV
        breakdown_path = results_dir / f"ensemble_holdout_category_breakdown_{timestamp}.csv"
        breakdown_df.to_csv(breakdown_path, index=False)
        print(f"✅ Holdout category breakdown saved to: {breakdown_path}")
        
        # Display in notebook
        display(breakdown_df)
        
    else:
        print("⚠️ No valid categorical features found in X_holdout for breakdown.")
        
    # 5. Save the full row-level predictions for deep-dive debugging
    row_level_path = results_dir / f"ensemble_holdout_row_level_predictions_{timestamp}.csv"
    eval_df.to_csv(row_level_path, index=False)
    print(f"\n✅ Full row-level holdout predictions (with true/pred labels) saved to: {row_level_path}")

else:
    print("⚠️ Holdout evaluation was not performed or X_holdout is unavailable.")

🚀 Generating holdout error analysis...
✅ Holdout category breakdown saved to: results/ensemble_holdout_category_breakdown_20260914_0335.csv


,feature_name,feature_value,total_samples,correct_predictions,wrong_predictions,accuracy,actual_yes_count,predicted_yes_count
0,Gender,Female,118054,105610,12444,0.894591,21048,21926
1,Gender,Male,147305,131932,15373,0.895638,25316,26751
2,Gender,Other,2107,1860,247,0.882772,348,311
3,City_Type,Rural,49626,43932,5694,0.885262,9669,10377
4,City_Type,Suburban,102135,91030,11105,0.891271,18429,19960
5,City_Type,Urban,115705,104440,11265,0.902640,18614,18651
6,Current_Car_Type,Hatchback,31784,28363,3421,0.892367,5664,5675
7,Current_Car_Type,SUV,98572,87843,10729,0.891156,17821,18960
8,Current_Car_Type,Sedan,121553,109197,12356,0.898349,20790,22052
9,Current_Car_Type,Truck,15557,13999,1558,0.899852,2437,2301



✅ Full row-level holdout predictions (with true/pred labels) saved to: results/ensemble_holdout_row_level_predictions_20260914_0335.csv
